In [ ]:
# ─── Standard Library ───────────────────────────────────────────────────────────
from pathlib import Path
import heapq

# ─── Scientific Computing ───────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ─── Geospatial Processing ──────────────────────────────────────────────────────
import geopandas as gpd
from shapely.geometry import box

# ─── Network Analysis ───────────────────────────────────────────────────────────
import pandana

# ─── Mapping and Visualization ──────────────────────────────────────────────────
import matplotlib.pyplot as plt

# ─── Progress Monitoring ────────────────────────────────────────────────────────
from tqdm.notebook import tqdm

# ─── Custom Modules ────────────────────────────────────────────────────────────
import geo_util as gu

In [ ]:
# global definitions
_data_path = Path( './data/' )

In [ ]:
net = pandana.Network.from_hdf5(_data_path.joinpath('nl_pandana_network.h5'))

In [ ]:
max_dist_threshold = 20_000  # in meters

In [ ]:
dist = pd.read_parquet(_data_path.joinpath(f'access_matrix_{max_dist_threshold}.parquet'))

In [ ]:
assert not dist.duplicated(subset=['pop_idx', 'poi_idx']).any()

In [ ]:
use_pop = gpd.read_parquet(_data_path.joinpath('population_aggregated_per_node.parquet'))

In [ ]:
use_banks = gpd.read_parquet(_data_path.joinpath('banks_aggregated_per_node.parquet'))

In [ ]:
dist_matrix = gu.SparseDistanceMatrix(
    dist,
    pop_col='pop_idx',
    poi_col='poi_idx',
    dist_col='total_dist'
)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
axes = axes.T  # So column-wise: axes[0] = left column, axes[1] = right column

columns = {
    'left': ['road_distance', 'total_dist'],
    'right': ['pop_to_node_dist', 'poi_to_node_dist']
}

for col_idx, (col_group, col_names) in enumerate(columns.items()):
    for row_idx, col_name in enumerate(col_names):
        gu.plotFastHistogram(
            dist[col_name].dropna().to_numpy(),
            ax=axes[col_idx][row_idx],
            title=col_name
        )

plt.tight_layout()
plt.show()


In [ ]:
m = gu.visualizeOrAddPopPoiConnection(
    row_idx=dist['total_dist'].idxmax(),
    all_distances=dist,
    population=use_pop,
    points_of_interest=use_banks,
    network=net,
    pop_id_col='pop_idx',
    poi_id_col='poi_idx'
)
m

In [ ]:
n = 5  # or whatever number you want
top_n_indices = dist['total_dist'].nlargest(n).index
for idx in top_n_indices[1:]:
    m = gu.visualizeOrAddPopPoiConnection(
        m=m,
        row_idx=idx,
        all_distances=dist,
        population=use_pop,
        points_of_interest=use_banks,
        network=net,
        pop_id_col='pop_idx',
        poi_id_col='poi_idx'
    )
m

In [ ]:
poi_pop = dist_matrix.poiToPopsWithin(1_000.0)

In [ ]:
min_keys = heapq.nsmallest(8, poi_pop, key=lambda k: len(poi_pop[k]))


In [ ]:
{ k : len(poi_pop[k]) for k in min_keys }

In [ ]:
poi = min_keys[-1]
pops = poi_pop[poi]

In [ ]:
# Suppose these are GeoDataFrame slices (one or multiple rows)
pop_rows = use_pop.loc[pops]
poi_rows = use_banks.loc[[poi]]

# Get the bounding box of all geometries
combined_bounds = pd.concat([pop_rows.geometry, poi_rows.geometry]).total_bounds
# => array: [minx, miny, maxx, maxy]

bbox = box(*combined_bounds)

In [ ]:
use_pop[use_pop.geometry.within(bbox)]

In [ ]:
if len(use_pop[use_pop.geometry.within(bbox)]) < 100:
    m = use_pop[use_pop.geometry.within(bbox)].explore(color='green',style_kwds={'radius': 8, 'weight': 1, 'fillOpacity': 0.9})
else:
    m = None
for pop in pops:
    row_idx = dist[(dist['poi_idx'] == poi) & (dist['pop_idx'] == pop)].index[0]
    m = gu.visualizeOrAddPopPoiConnection(
        m=m,
        row_idx=row_idx,
        all_distances=dist,
        population=use_pop,
        points_of_interest=use_banks,
        network=net,
        pop_id_col='pop_idx',
        poi_id_col='poi_idx'
    )
m

In [ ]:
def computeCoverageSeries(
    dist_matrix,
    use_pop: pd.DataFrame,
    max_dist_threshold: int,
    step: int = 500
) -> dict[int, float]:
    """
    Computes coverage percentage for a range of distance thresholds,
    avoiding double counting population covered by multiple POIs.

    Args:
        dist_matrix: Object with poiToPopsWithin(meters) method.
        use_pop: DataFrame with 'pop_idx' and 'Population' columns.
        max_dist_threshold: Maximum distance (in meters) to evaluate.
        step: Step size between distance thresholds (default: 500).

    Returns:
        Dictionary mapping distance (in meters) to population coverage (%).
    """
    total_population = use_pop['Population'].sum()
    pop_map = use_pop.set_index('pop_idx')['Population']

    def coverage(meters: int) -> float:
        poi_pop = dist_matrix.poiToPopsWithin(meters)
        if not poi_pop:
            return 0.0
        covered_ids = np.unique(np.concatenate(list(poi_pop.values())))
        return pop_map.loc[covered_ids].sum() / total_population * 100

    return {
        m: coverage(m)
        for m in tqdm(range(0, max_dist_threshold + 1, step), desc='Evaluating coverage')
    }


In [ ]:
max_cov = computeCoverageSeries( dist_matrix, use_pop, max_dist_threshold, 500 )

In [ ]:
# Convert to sorted x and y lists
x = sorted(max_cov.keys())
y = [max_cov[k] for k in x]

# Plot
plt.figure(figsize=(10, 5))
plt.plot(x, y, marker='o')
plt.xlabel('meters')
plt.ylabel('coverage')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
pd.DataFrame.from_dict( max_cov, orient='index' )

In [ ]:
covered = dist.pop_idx.unique()
sum_covered = use_pop.loc[dist.pop_idx.unique()].Population.sum()

In [ ]:
use_pop.Population.sum()-sum_covered

In [ ]:
import folium
from shapely.geometry import LineString
from pyproj import Transformer
import pandas as pd
import geopandas as gpd
import pandana as pdna

def addPopPoiPathToMap(
    pop_idx: int,
    poi_idx: int,
    population: gpd.GeoDataFrame,
    pois: gpd.GeoDataFrame,
    network: pdna.Network,
    m: folium.Map = None,
    pop_node_col: str = 'nearest_node_id',
    poi_node_col: str = 'nearest_node_id',
    metric_crs: str = 'EPSG:28992',
    color: str = 'red',
    weight: int = 3
) -> folium.Map:
    """
    Adds a population–POI path and snapping lines to a Folium map using a Pandana network.

    Args:
        pop_idx: ID of the population point (matches `population.pop_idx`).
        poi_idx: ID of the POI (matches `pois.poi_idx`).
        population: GeoDataFrame with population points and nearest Pandana node.
        pois: GeoDataFrame with POI points and nearest Pandana node.
        network: Pandana network object.
        m: Optional existing folium.Map to add to. If None, creates a new one.
        pop_node_col: Column in `population` with Pandana node IDs.
        poi_node_col: Column in `pois` with Pandana node IDs.
        metric_crs: CRS in which geometries and distances are defined.
        color: Line color for the shortest path.
        weight: Line thickness.

    Returns:
        folium.Map with path and snapping lines added.
    """
    pop_row = population.loc[population.pop_idx == pop_idx].iloc[0]
    poi_row = pois.loc[pois.poi_idx == poi_idx].iloc[0]

    from_node = pop_row[pop_node_col]
    to_node = poi_row[poi_node_col]
    path_nodes = network.shortest_path(from_node, to_node)

    if path_nodes is None or len(path_nodes) < 2:
        print('no path')
        return m

    transformer = Transformer.from_crs(metric_crs, 'EPSG:4326', always_xy=True)

    # Path line
    path_coords = network.nodes_df.loc[path_nodes, ['x', 'y']].to_numpy()
    path_coords_lonlat = [transformer.transform(x, y) for x, y in path_coords]
    path_length = LineString(path_coords).length

    # Geometry for snapping lines
    snap_pop_xy = network.nodes_df.loc[from_node, ['x', 'y']]
    snap_poi_xy = network.nodes_df.loc[to_node, ['x', 'y']]
    snap_pop_lonlat = transformer.transform(snap_pop_xy['x'], snap_pop_xy['y'])
    snap_poi_lonlat = transformer.transform(snap_poi_xy['x'], snap_poi_xy['y'])

    pop_geom = pop_row.geometry
    poi_geom = poi_row.geometry
    pop_lonlat = transformer.transform(pop_geom.x, pop_geom.y)
    poi_lonlat = transformer.transform(poi_geom.x, poi_geom.y)

    # New map if needed
    if m is None:
        center_latlon = [(pop_lonlat[1] + poi_lonlat[1]) / 2, (pop_lonlat[0] + poi_lonlat[0]) / 2]
        m = folium.Map(location=center_latlon, zoom_start=13)

    # Markers
    folium.Marker(location=(pop_lonlat[1], pop_lonlat[0]), tooltip=f'Pop {pop_idx}').add_to(m)
    folium.Marker(location=(poi_lonlat[1], poi_lonlat[0]), tooltip=f'POI {poi_idx}').add_to(m)

    # Snap points
    folium.CircleMarker(location=(snap_pop_lonlat[1], snap_pop_lonlat[0]), radius=4, color='blue', fill=True).add_to(m)
    folium.CircleMarker(location=(snap_poi_lonlat[1], snap_poi_lonlat[0]), radius=4, color='green', fill=True).add_to(m)

    # Snap lines
    folium.PolyLine([(pop_lonlat[1], pop_lonlat[0]), (snap_pop_lonlat[1], snap_pop_lonlat[0])],
                    color='black', dash_array='3', tooltip='Snap (pop)').add_to(m)
    folium.PolyLine([(poi_lonlat[1], poi_lonlat[0]), (snap_poi_lonlat[1], snap_poi_lonlat[0])],
                    color='black', dash_array='3', tooltip='Snap (poi)').add_to(m)

    # Path line
    folium.PolyLine([(lat, lon) for lon, lat in path_coords_lonlat],
                    color=color, weight=weight, tooltip=f'Shortest path: {path_length:.1f} m').add_to(m)

    return m


In [ ]:
import networkx as nx
G = nx.from_pandas_edgelist(net.edges_df, source='from', target='to', create_using=nx.Graph)
cm = {
    node: idx
    for idx, component in enumerate(nx.connected_components(G))
    for node in component
}

In [ ]:
use_pop['component'] = use_pop.nearest_node_id.map(cm)
use_banks['component'] = use_banks.nearest_node_id.map(cm)

In [ ]:
m = use_banks.explore(color='red')
use_pop[~use_pop.pop_idx.isin(covered)].explore(m=m,color='blue')

In [ ]:
pop_idx=1778208
poi_idx=171
addPopPoiPathToMap(
    pop_idx=pop_idx,
    poi_idx=poi_idx,
    population=use_pop,
    pois=use_banks,
    network=net
)

In [ ]:
pop_row = use_pop.loc[use_pop.pop_idx == pop_idx].iloc[0]
poi_row = use_banks.loc[use_banks.poi_idx == poi_idx].iloc[0]

from_node = pop_row.nearest_node_id
to_node = poi_row.nearest_node_id

cm[from_node], cm[to_node]


In [ ]:
isolated = use_pop[~use_pop.pop_idx.isin(covered)]

In [ ]:
from collections import Counter

Counter(cm[n] for n in isolated.nearest_node_id if n in cm)